# Hierarchical Fragment Linker 結果視覺化

使用 `HierarchicalFragmentLinker` 從原始三張輸入（image / mask / annotation）進行完整流程（預處理 + 階層式重建）並視覺化結果。

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import networkx as nx

from neural_reconstruction.algorithms.fragment_linking.linker import HierarchicalFragmentLinker

In [ ]:
IMAGE_ID = 'S1585-2_a'
BASE_PATH = f'../data/{IMAGE_ID}'

## 1. 載入原始輸入

In [ ]:
image      = cv2.imread(f'{BASE_PATH}/image.png',      cv2.IMREAD_COLOR_RGB)[:, :, 1] 
mask       = cv2.imread(f'{BASE_PATH}/mask.png',       cv2.IMREAD_GRAYSCALE)
annotation = cv2.imread(f'{BASE_PATH}/annotation.png', cv2.IMREAD_GRAYSCALE)

print(f'image      shape: {image.shape},      dtype: {image.dtype}')
print(f'mask       shape: {mask.shape},       dtype: {mask.dtype}')
print(f'annotation shape: {annotation.shape}, dtype: {annotation.dtype}')

_, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
axes[0].imshow(image,      cmap='gray');  axes[0].set_title('Image');       axes[0].axis('off')
axes[1].imshow(mask,       cmap='gray');  axes[1].set_title('Mask');        axes[1].axis('off')
axes[2].imshow(annotation, cmap='gray');  axes[2].set_title('Annotation');  axes[2].axis('off')
plt.show()

## 2. 執行 HierarchicalFragmentLinker（含預處理）

In [ ]:
linker = HierarchicalFragmentLinker(
    # 預處理參數
    offset_px=100,
    rolling_ball_radius=1,
    sato_weight=1,
    opening_kernel_size=3,
    # 種子圖參數
    segment_length=3.0,
    # 路徑查找參數
    search_radius_pathfinding=20.0,
    # 階段1：端點延伸
    search_radius_endpoint_extension=10.0,
    max_angle_endpoint_extension=75.0,
    angle_penalty_endpoint_extension=0.5,
    direction_threshold_endpoint_extension=5.0,
    # 階段2：MST 候選邊
    search_radius_mst=20.0,
    max_angle_mst=90.0,
    angle_penalty_mst=0.5,
    distance_weight_mst=0.2,
    max_cost_threshold_mst=0.75,
    # MST 折扣
    endpoint_extension_weight_discount=0.5,
)

result_graph = linker.run(image, mask, annotation)

print(f'Nodes: {result_graph.number_of_nodes()}')
print(f'Edges: {result_graph.number_of_edges()}')
print(f'Connected components: {nx.number_connected_components(result_graph)}')

## 3. 視覺化結果

### 3.1 全圖覽 — Annotation vs 重建結果

In [ ]:
def draw_graph_on_ax(ax, background, graph, edge_color='lime', node_color='cyan',
                     linewidth=1, node_size=1.2, show_nodes=True):
    ax.imshow(background, cmap='gray' if background.ndim == 2 else None)
    for u, v, data in graph.edges(data=True):
        path = data.get('path', [u, v])
        if len(path) < 2:
            continue
        ax.plot([p[1] for p in path], [p[0] for p in path],
                color=edge_color, linewidth=linewidth, alpha=0.85)
    if show_nodes:
        for node in graph.nodes():
            ax.plot(node[1], node[0], '.', color=node_color, markersize=node_size)


fig, axes = plt.subplots(2, 1, figsize=(128, 128), constrained_layout=True)

axes[0].imshow(image, cmap='gray')
axes[0].imshow(np.ma.masked_where(annotation == 0, annotation), cmap='Reds', alpha=0.7)
axes[0].set_title('原始 Annotation')
axes[0].axis('off')

draw_graph_on_ax(axes[1], image, result_graph)
axes[1].set_title(f'Hierarchical Fragment Linking 重建  (nodes={result_graph.number_of_nodes()}, edges={result_graph.number_of_edges()})')
axes[1].axis('off')

plt.show()

### 3.2 各連通分量以不同顏色標示

In [ ]:
components = list(nx.connected_components(result_graph))
cmap_comp  = plt.cm.get_cmap('tab20', max(len(components), 1))

fig, ax = plt.subplots(figsize=(16, 10), constrained_layout=True)
ax.imshow(image, cmap='gray')

for idx, comp_nodes in enumerate(components):
    color    = cmap_comp(idx % 20)
    subgraph = result_graph.subgraph(comp_nodes)
    for u, v, data in subgraph.edges(data=True):
        path = data.get('path', [u, v])
        if len(path) >= 2:
            ax.plot([p[1] for p in path], [p[0] for p in path],
                    color=color, linewidth=1.5, alpha=0.9)
    for node in comp_nodes:
        ax.plot(node[1], node[0], '.', color=color, markersize=3)

ax.set_title(f'連通分量視覺化  ({len(components)} 個分量)')
ax.axis('off')
plt.show()

### 3.3 邊權重 (Cost) 熱圖

In [ ]:
weights = [d.get('weight', 0) for _, _, d in result_graph.edges(data=True)]

if weights:
    w_min, w_max = min(weights), max(weights)
    norm      = plt.Normalize(vmin=w_min, vmax=w_max)
    cmap_cost = plt.cm.viridis

    fig, ax = plt.subplots(figsize=(16, 10), constrained_layout=True)
    ax.imshow(image, cmap='gray')

    for u, v, data in result_graph.edges(data=True):
        path = data.get('path', [u, v])
        if len(path) < 2:
            continue
        color = cmap_cost(norm(data.get('weight', 0)))
        ax.plot([p[1] for p in path], [p[0] for p in path],
                color=color, linewidth=1.5, alpha=0.9)

    sm = plt.cm.ScalarMappable(cmap=cmap_cost, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Edge Weight (Cost)', fraction=0.03, pad=0.02)
    ax.set_title('邊權重分布')
    ax.axis('off')
    plt.show()

    print(f'Weight stats — min: {w_min:.4f}, max: {w_max:.4f}, mean: {np.mean(weights):.4f}')
else:
    print('No edges to visualize.')

### 3.4 三欄比較：原始影像 / Annotation / 重建結果

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(27, 9), constrained_layout=True)

axes[0].imshow(image, cmap='gray')
axes[0].set_title('原始影像')
axes[0].axis('off')

axes[1].imshow(image, cmap='gray')
axes[1].imshow(np.ma.masked_where(annotation == 0, annotation), cmap='Reds', alpha=0.75)
axes[1].set_title('原始 Annotation')
axes[1].axis('off')

draw_graph_on_ax(axes[2], image, result_graph, edge_color='lime', node_color='yellow',
                 linewidth=1.5, node_size=2)
axes[2].set_title('Hierarchical Fragment Linking 重建結果')
axes[2].axis('off')

plt.show()

## 4. 統計摘要

In [ ]:
print('=== HierarchicalFragmentLinker 執行結果 ===')
print(f'影像尺寸: {image.shape}')
print(f'Nodes (種子點): {result_graph.number_of_nodes()}')
print(f'Edges (連接邊): {result_graph.number_of_edges()}')
print(f'連通分量數:     {nx.number_connected_components(result_graph)}')

phase1_edges = sum(1 for _, _, d in result_graph.edges(data=True) if d.get('phase') == 1)
phase2_edges = sum(1 for _, _, d in result_graph.edges(data=True) if d.get('phase') == 2)
print(f'\n=== 階段統計 ===')
print(f'Phase 1 edges (端點延伸): {phase1_edges}')
print(f'Phase 2 edges (MST 候選): {phase2_edges}')

if weights:
    print(f'\n=== 邊權重統計 ===')
    print(f'Min:  {w_min:.4f}')
    print(f'Max:  {w_max:.4f}')
    print(f'Mean: {np.mean(weights):.4f}')
    print(f'Std:  {np.std(weights):.4f}')

    plt.figure(figsize=(8, 4))
    plt.hist(weights, bins=40, color='steelblue', edgecolor='white')
    plt.xlabel('Edge Weight (Cost)')
    plt.ylabel('Count')
    plt.title('邊權重分布直方圖')
    plt.tight_layout()
    plt.show()

## 5. Average Hausdorff Distance（與 GT 比較）

In [ ]:
from neural_reconstruction.core.topology import TopologyBuilder
from neural_reconstruction.core.evaluation import extract_graph_points, compute_average_hausdorff_distance

# 從 label.png 建構 GT 拓樸
label_img = cv2.imread(f'{BASE_PATH}/label.png', cv2.IMREAD_GRAYSCALE)

gt_builder = TopologyBuilder()
gt_graph = gt_builder.build_seed_graph(label_img)

print(f'GT  — Nodes: {gt_graph.number_of_nodes()}, Edges: {gt_graph.number_of_edges()}')
print(f'Pred — Nodes: {result_graph.number_of_nodes()}, Edges: {result_graph.number_of_edges()}')

# 展開成點集（nodes + edge path 上的所有點）
pred_points = extract_graph_points(result_graph)
gt_points   = extract_graph_points(gt_graph)

print(f'\nPoint count after expanding edge paths:')
print(f'  Pred: {len(pred_points)} pts')
print(f'  GT  : {len(gt_points)} pts')

In [ ]:
avg_dist, d_pred_to_gt, d_gt_to_pred = compute_average_hausdorff_distance(
    pred_points, gt_points, return_components=True
)

print('=== Average Hausdorff Distance ===')
print(f'Avg Hausdorff Distance : {avg_dist:.4f} px')
print(f'  Pred → GT            : {d_pred_to_gt:.4f} px')
print(f'  GT   → Pred          : {d_gt_to_pred:.4f} px')

In [ ]:
from neural_reconstruction.core.evaluation import compute_point_min_distances

min_pred_to_gt, min_gt_to_pred = compute_point_min_distances(pred_points, gt_points)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

BINS = 50

for ax, dists, label, color in [
    (axes[0], min_pred_to_gt, 'Pred → GT', 'steelblue'),
    (axes[1], min_gt_to_pred, 'GT → Pred', 'tomato'),
]:
    counts, edges = np.histogram(dists, bins=BINS)
    # 由距離高到低：反轉 bin 順序
    centers = (edges[:-1] + edges[1:]) / 2
    order   = np.argsort(centers)[::-1]
    ax.bar(range(BINS), counts[order], color=color, edgecolor='white', linewidth=0.4)
    ax.set_xticks(range(0, BINS, 5))
    ax.set_xticklabels([f'{centers[order][i]:.1f}' for i in range(0, BINS, 5)], rotation=45, ha='right')
    ax.set_xlabel('最小距離 (px)  ← 高到低')
    ax.set_ylabel('點數')
    ax.set_title(f'{label}\n'
                 f'mean={np.mean(dists):.2f}  median={np.median(dists):.2f}  '
                 f'max={np.max(dists):.2f}  px')

fig.suptitle('每個點到對方點集的最小距離分布（由高到低排列）', fontsize=13)
plt.show()

# 累積分布（ECDF），方便確認多少比例的點誤差在閾值以內
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for dists, label, color in [
    (min_pred_to_gt, 'Pred → GT', 'steelblue'),
    (min_gt_to_pred, 'GT → Pred', 'tomato'),
]:
    sorted_d = np.sort(dists)[::-1]          # 由高到低
    ecdf     = np.arange(1, len(sorted_d)+1) / len(sorted_d)
    ax.plot(sorted_d, ecdf, label=label, color=color)

ax.set_xlabel('最小距離閾值 (px)')
ax.set_ylabel('比例（≥ 該距離的點）')
ax.set_title('ECDF — 由高到低（顯示多少比例的點距離超過 x px）')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()